<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5512-notebooks/main/_banners/opim5512_banner.svg" alt="OPIM 5512 banner" width="100%">

# Energy Project Setup Guide (Fall 2026)
### OPIM 5512 · Module 4 · Lab 5 pre-lab · about 40 minutes, once

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5512-notebooks/blob/main/Module4/Energy_Project_Setup_F26.ipynb)

***Do NOT click Run all.** Copy each gray cell into **Cloud Shell** (in the Google Cloud Console), one at a time, top to bottom.*

## What this is
These are **the same steps you already did** for your Craigslist project, in a **new** Google Cloud project, for a **new**
repo: [`drdave-teaching/energy-forecasting`](https://github.com/drdave-teaching/energy-forecasting). Your Craigslist project
keeps running exactly as it is.

**At the end you'll have four Cloud Functions running every hour in your energy project:**

| Minute | Function | What it saves in `gs://YOUR-BUCKET/energy/` |
| --- | --- | --- |
| :05 | `scraper-weather-obs` | the weather that actually happened at Bradley Airport |
| :05 | `scraper-weather-forecast` | the National Weather Service forecast for Bradley, next 24 hours |
| :15 | `scraper-isone` | Connecticut electricity load from ISO New England, plus ISO New England's own forecast |
| :20 | `forecast-24h` | your model's forecast for the next 24 hours (it waits politely until Lab 5 gives it a model) |

**Your checklist** (tick them off as you go):

| # | Step | Where | Time |
|---|---|---|---|
| 0 | New project, budget, fork, ISO Express account, open Cloud Shell | Browser | 10 min |
| 1 | Paste your 4 variables | Cloud Shell | 1 min |
| 2 | Turn on the Google Cloud services | Cloud Shell | 2 min |
| 3 | Create the three robot accounts | Cloud Shell | 1 min |
| 4 | Connect GitHub to Google Cloud | Cloud Shell | 1 min |
| 5 | Give the robots their permissions | Cloud Shell | 2 min |
| 6 | Create your bucket | Cloud Shell | 1 min |
| 7 | Check everything | Cloud Shell | 1 min |
| 8 | Add 6 variables and 2 secrets to GitHub | GitHub | 5 min |
| 9 | Deploy the four functions and watch the bucket fill | GitHub + Console | 15 min |

## 🤖 Stuck? Ask Claude
**Want a guide at your side the whole way?** Paste this into Claude before you start:

```
Here is my class's Google Cloud setup guide: https://github.com/drdave-teaching/OPIM5512-notebooks/blob/main/Module4/Energy_Project_Setup_F26.ipynb
Walk me through it one step at a time. Tell me exactly what to paste into Cloud Shell or click
in GitHub, wait for me to paste back what happened, and help me fix any errors before we move on.
```

When something goes red, copy this prompt into Claude, fill in the blanks, and paste the **whole** error (a screenshot works too):

```
I'm setting up an hourly weather and electricity-load pipeline on Google Cloud, deployed with
GitHub Actions, for my OPIM 5512 class. I'm on Step ___ of the energy setup guide.

I ran this in Cloud Shell (or: this GitHub Actions workflow failed):
___ paste the command or the workflow name ___

This is the full error:
___ paste the error ___

My project ID is ___ and my GitHub repo is ___/___.
What does this error mean, and what exactly should I type or click next?
```

Two rules: do the steps **in order** (most errors come from a skipped step, or from a Cloud Shell restart that wiped your
variables), and **never paste passwords, keys, or tokens** into any chat. That includes your ISO Express password: it goes
only into GitHub's Secrets box in Step 8.

Also check the **Troubleshooting** table at the very bottom: every error we hit while testing this guide is in it.

# Step 0 · Before you start (browser, 10 minutes)

### 0a. Create your energy project
1. Go to **console.cloud.google.com** with the **same personal Gmail** as your Craigslist project.
2. Click the project picker at the top, then **New project**. Name it, for example, `energy-yournetid-f26`.
3. Look at the grey **Project ID** under the name box and write it down: that ID is what you paste in Step 1. (Google may
   add a number, like `energy-dww05002-f26-123456`. Copy it exactly.) Leave **Organization** as **No organization**. Click **Create**.
4. **Important:** the console stays on your *old* project after it creates the new one. Click the project picker again
   and **select your energy project**.
5. Check billing: search **Billing** → **Account management**. Your energy project should be listed. (On the free trial
   it links automatically.)

### 0b. Set a budget (1 minute, don't skip)
Search **Budgets & alerts** → **Create budget**. Scope: this project, all services. Amount: `300`. Keep the 50 / 90 / 100 %
alerts. **Finish**. These functions run once an hour and save small files, so the project should cost pennies. A budget only
sends emails; if you ever get one, stop and ask for help.

### 0c. Fork the energy repo
1. Open **github.com/drdave-teaching/energy-forecasting** and click **Fork** (you may rename it, for example
   `energy-forecasting-f26`). Under the name it must say *"forked from drdave-teaching/energy-forecasting"*.
2. In **your fork**, open the **Actions** tab. GitHub shows **"Workflows aren't being run on this forked repository."**
   Click the green **"I understand my workflows, go ahead and enable them."** (Skip this and Step 9 has nothing to run.)
3. Copy your fork's name from the address bar, as `your-github-username/your-repo-name`. You paste it in Step 1.

### 0d. Get your own ISO Express account (free, 5 minutes)
ISO New England runs Connecticut's power grid and publishes its load data through **ISO Express web services**, which need
a free login. Create **your own** account at [iso-ne.com](https://www.iso-ne.com/) (look for **Log in / Register** at the top,
then the ISO Express / web services registration). Never use Dr. Dave's or a classmate's account. Keep your username and
password handy for Step 8, and paste them nowhere else.

### 0e. Open Cloud Shell
With your **energy** project selected at the top of the console, click the **`>_`** icon at the top right. The terminal opens
at the bottom of the page. The first time you run a command, Google asks to **Authorize Cloud Shell**: click **Authorize**.

**If Cloud Shell disconnects** (it times out when idle), your variables are wiped. Paste the Step 1 cell again before
anything else.

# Step 1 · Your 4 variables (Cloud Shell, 1 minute)
Copy this cell into a text editor first and change the **two lines marked CHANGE ME** (project ID twice, and your fork).
Then paste the whole cell into Cloud Shell and press Enter.

In [ ]:
# ==================== CHANGE THE TWO "CHANGE ME" LINES ====================
PROJECT_ID="YOUR-ENERGY-PROJECT-ID"           # CHANGE ME, e.g. "energy-dww05002-f26"
GITHUB_REPO="YOUR-GITHUB-USER/YOUR-REPO"      # CHANGE ME, e.g. "drdww/energy-forecasting"
BUCKET_NAME="${PROJECT_ID}"                   # leave this: your bucket has the same name as your project
REGION="us-central1"                          # leave this
# =================== leave everything below alone ========================
WIF_POOL="github-pool"
WIF_PROVIDER="gh-actions"
RUNTIME_SA_ID="cf-runtime"
DEPLOYER_SA_ID="cf-deployer"
SCHED_SA_ID="scheduler-invoker"
gcloud config set project "${PROJECT_ID}"
PROJECT_NUMBER="$(gcloud projects describe "${PROJECT_ID}" --format='value(projectNumber)')"
RUNTIME_SA="${RUNTIME_SA_ID}@${PROJECT_ID}.iam.gserviceaccount.com"
DEPLOYER_SA="${DEPLOYER_SA_ID}@${PROJECT_ID}.iam.gserviceaccount.com"
SCHED_SA="${SCHED_SA_ID}@${PROJECT_ID}.iam.gserviceaccount.com"
SCHEDULER_AGENT="service-${PROJECT_NUMBER}@gcp-sa-cloudscheduler.iam.gserviceaccount.com"
PRINCIPAL_SET="principalSet://iam.googleapis.com/projects/${PROJECT_NUMBER}/locations/global/workloadIdentityPools/${WIF_POOL}/attribute.repository/${GITHUB_REPO}"
echo "✅ Project: ${PROJECT_ID} (number: ${PROJECT_NUMBER})"
echo "✅ GitHub repo: ${GITHUB_REPO}"

**What you should see:** two ✅ lines, and the project **number** is a long number, like:
```
✅ Project: energy-dww05002-f26 (number: 866788805467)
✅ GitHub repo: drdww/energy-forecasting
```
**Number blank** (`number: )`)? The project ID is mistyped, or you haven't clicked **Authorize** yet. Fix it and paste the cell again.
Don't continue until both lines look right.

# Step 2 · Turn on the Google Cloud services (Cloud Shell, about 2 minutes)
Your project starts with almost everything switched off. This turns on Cloud Functions, Cloud Scheduler, Cloud Storage and
the services they depend on. Safe to run twice.

In [ ]:
gcloud services enable cloudfunctions.googleapis.com run.googleapis.com cloudscheduler.googleapis.com \
  artifactregistry.googleapis.com iam.googleapis.com iamcredentials.googleapis.com cloudbuild.googleapis.com \
  eventarc.googleapis.com storage.googleapis.com serviceusage.googleapis.com cloudresourcemanager.googleapis.com \
  pubsub.googleapis.com logging.googleapis.com compute.googleapis.com \
  && echo "✅ Step 2 done: services are on"

**What you should see** (after a minute or two of nothing, which is normal):
```
Operation "operations/acf.p2-...." finished successfully.
✅ Step 2 done: services are on
```

# Step 3 · Create the three robot accounts (Cloud Shell, 1 minute)
Google calls them **service accounts**: logins for programs, not people.
- `cf-runtime`: your Cloud Functions run as this account (it reads and writes your bucket).
- `cf-deployer`: GitHub uses this account to deploy your functions.
- `scheduler-invoker`: Cloud Scheduler uses it to start your functions every hour.

In [ ]:
gcloud iam service-accounts create "${RUNTIME_SA_ID}"  --display-name="Cloud Functions Runtime" || true
gcloud iam service-accounts create "${DEPLOYER_SA_ID}" --display-name="GitHub Deployer" || true
gcloud iam service-accounts create "${SCHED_SA_ID}"    --display-name="Scheduler Invoker" || true
echo "✅ Step 3 done"

**What you should see:** three `Created service account [...]` lines and `✅ Step 3 done`. If you run it a second time it says
`already exists`: that's fine.

# Step 4 · Connect GitHub to Google Cloud (Cloud Shell, 1 minute)
**Workload Identity Federation** lets GitHub Actions log in to your project without any password or key file. It only
trusts **your fork's `main` branch**, which is why `GITHUB_REPO` in Step 1 must match your fork exactly.

In [ ]:
gcloud iam workload-identity-pools create "${WIF_POOL}" --location="global" --display-name="GitHub Actions Pool" || true
gcloud iam workload-identity-pools providers create-oidc "${WIF_PROVIDER}" \
  --workload-identity-pool="${WIF_POOL}" --location="global" --display-name="GitHub OIDC Provider" \
  --issuer-uri="https://token.actions.githubusercontent.com" \
  --attribute-mapping="google.subject=assertion.sub,attribute.actor=assertion.actor,attribute.repository=assertion.repository,attribute.ref=assertion.ref" \
  --attribute-condition="attribute.repository=='${GITHUB_REPO}' && attribute.ref=='refs/heads/main'" || true
gcloud iam service-accounts add-iam-policy-binding "${DEPLOYER_SA}" \
  --role="roles/iam.workloadIdentityUser" --member="${PRINCIPAL_SET}" >/dev/null \
  && echo "✅ Step 4 done: GitHub (${GITHUB_REPO}) can use ${DEPLOYER_SA}"

**What you should see:**
```
Created workload identity pool [github-pool].
Created workload identity pool provider [gh-actions].
Updated IAM policy for serviceAccount [cf-deployer@...].
✅ Step 4 done: GitHub (your-user/your-repo) can use cf-deployer@...
```

# Step 5 · Give the robots their permissions (Cloud Shell, about 2 minutes)
Each line prints a ✅ when it works. Nine lines in total.

In [ ]:
for ROLE in roles/cloudfunctions.developer roles/run.admin roles/cloudscheduler.admin roles/artifactregistry.writer roles/serviceusage.serviceUsageAdmin; do
  gcloud projects add-iam-policy-binding "${PROJECT_ID}" --member="serviceAccount:${DEPLOYER_SA}" --role="${ROLE}" --condition=None >/dev/null \
    && echo "✅ deployer can: ${ROLE}"
done
gcloud iam service-accounts add-iam-policy-binding "${RUNTIME_SA}" --member="serviceAccount:${DEPLOYER_SA}" --role="roles/iam.serviceAccountUser" >/dev/null \
  && echo "✅ deployer can run things as cf-runtime"
gcloud iam service-accounts add-iam-policy-binding "${RUNTIME_SA}" --member="serviceAccount:${DEPLOYER_SA}" --role="roles/iam.serviceAccountAdmin" >/dev/null \
  && echo "✅ deployer can manage cf-runtime"
gcloud iam service-accounts add-iam-policy-binding "projects/-/serviceAccounts/${PROJECT_NUMBER}-compute@developer.gserviceaccount.com" \
  --member="serviceAccount:${DEPLOYER_SA}" --role="roles/iam.serviceAccountUser" >/dev/null \
  && echo "✅ deployer can use the default compute account"
gcloud iam service-accounts add-iam-policy-binding "${RUNTIME_SA}" --member="serviceAccount:${SCHED_SA}" --role="roles/iam.serviceAccountTokenCreator" >/dev/null \
  && echo "✅ scheduler-invoker can start your functions"
gcloud iam service-accounts add-iam-policy-binding "${RUNTIME_SA}" --member="serviceAccount:${SCHEDULER_AGENT}" --role="roles/iam.serviceAccountTokenCreator" >/dev/null \
  && echo "✅ Cloud Scheduler can start your functions"
echo "Step 5 finished: count the ✅ lines above (9 is perfect)"

**What you should see:** nine ✅ lines. If **only the last one** fails with *"service account ... does not exist"*, that's
OK: Google creates the Cloud Scheduler account a little later, and the deploy workflow in Step 9 grants that permission
automatically. Any other red line: paste the cell again (permissions sometimes take a few seconds to settle), then ask Claude.

# Step 6 · Create your bucket (Cloud Shell, 1 minute)
Your bucket is where every scraper and your model save their files. Same name as your project.

In [ ]:
if ! gcloud storage buckets describe "gs://${BUCKET_NAME}" >/dev/null 2>&1; then
  gcloud storage buckets create "gs://${BUCKET_NAME}" --location="${REGION}"
fi
gcloud storage buckets add-iam-policy-binding "gs://${BUCKET_NAME}" \
  --member="serviceAccount:${RUNTIME_SA}" --role="roles/storage.objectAdmin" >/dev/null \
  && echo "✅ Step 6 done: gs://${BUCKET_NAME} exists and cf-runtime can write to it"

**What you should see:** `Creating gs://your-project-id/...` and `✅ Step 6 done`.

# Step 7 · Check everything (Cloud Shell, 1 minute)
One read-only check of Steps 3 to 6. Nothing changes.

In [ ]:
echo "DEPLOYER PROJECT ROLES (expect 5):"
gcloud projects get-iam-policy "${PROJECT_ID}" --flatten="bindings[].members" --filter="bindings.members:${DEPLOYER_SA}" --format="value(bindings.role)"
echo; echo "GITHUB LINK (expect workloadIdentityUser + your repo):"
gcloud iam service-accounts get-iam-policy "${DEPLOYER_SA}" --format="value(bindings.role,bindings.members)"
echo; echo "CF-RUNTIME PERMISSIONS (expect 4 lines):"
gcloud iam service-accounts get-iam-policy "${RUNTIME_SA}" --flatten="bindings[].members" --format="value(bindings.role,bindings.members)"
echo; echo "BUCKET:"
gcloud storage buckets describe "gs://${BUCKET_NAME}" --format="value(name,location)"

**What you should see** (your project ID and number in place of these):
```
DEPLOYER PROJECT ROLES (expect 5):
roles/artifactregistry.writer
roles/cloudfunctions.developer
roles/cloudscheduler.admin
roles/run.admin
roles/serviceusage.serviceUsageAdmin

GITHUB LINK (expect workloadIdentityUser + your repo):
roles/iam.workloadIdentityUser  ['principalSet://iam.googleapis.com/projects/866788805467/.../attribute.repository/drdww/energy-forecasting']

CF-RUNTIME PERMISSIONS (expect 4 lines):
roles/iam.serviceAccountAdmin          serviceAccount:cf-deployer@energy-dww05002-f26.iam.gserviceaccount.com
roles/iam.serviceAccountTokenCreator   serviceAccount:scheduler-invoker@energy-dww05002-f26.iam.gserviceaccount.com
roles/iam.serviceAccountTokenCreator   serviceAccount:service-866788805467@gcp-sa-cloudscheduler.iam.gserviceaccount.com
roles/iam.serviceAccountUser           serviceAccount:cf-deployer@energy-dww05002-f26.iam.gserviceaccount.com

BUCKET:
energy-dww05002-f26    US-CENTRAL1
```
Check that the repo at the end of the GITHUB LINK line is **exactly your fork**. (Only 3 cf-runtime lines? That's the
Cloud Scheduler line from Step 5; Step 9 adds it.)

# Step 8 · Add 6 variables and 2 secrets to GitHub (5 minutes)

**8a. Print your values.** Paste this into Cloud Shell. It prints the 6 variables with your values filled in, ready to copy.

In [ ]:
WORKLOAD_IDENTITY_PROVIDER="projects/${PROJECT_NUMBER}/locations/global/workloadIdentityPools/${WIF_POOL}/providers/${WIF_PROVIDER}"
if [ -z "${PROJECT_NUMBER}" ]; then echo "⚠️  Your project number is blank: paste the Step 1 cell again, then this cell."; fi
echo "=================================================================="
echo "GitHub > your fork > Settings > Secrets and variables > Actions > VARIABLES tab"
echo "=================================================================="
printf "%-28s %s\n" "BUCKET_NAME"                "${BUCKET_NAME}"
printf "%-28s %s\n" "DEPLOYER_SA"                "${DEPLOYER_SA}"
printf "%-28s %s\n" "PROJECT_ID"                 "${PROJECT_ID}"
printf "%-28s %s\n" "REGION"                     "${REGION}"
printf "%-28s %s\n" "RUNTIME_SA"                 "${RUNTIME_SA}"
printf "%-28s %s\n" "WORKLOAD_IDENTITY_PROVIDER" "${WORKLOAD_IDENTITY_PROVIDER}"

**8b. Add the 6 variables.** In **your energy fork** on GitHub: **Settings** → scroll the left sidebar down to
**Security and quality** → **Secrets and variables** → **Actions** → open the **Variables** tab → click
**New repository variable** once per row, copying the name and value from the printout:

| Name | Value |
| --- | --- |
| `BUCKET_NAME` | from the printout |
| `DEPLOYER_SA` | from the printout |
| `PROJECT_ID` | from the printout |
| `REGION` | `us-central1` |
| `RUNTIME_SA` | from the printout |
| `WORKLOAD_IDENTITY_PROVIDER` | from the printout (the long `projects/…/providers/gh-actions` line) |

No `BASE_SITE` this time: that was only for Craigslist. (**Settings → Actions**, higher up in the sidebar, is a different page:
your variables are not there.)

**8c. Add your 2 ISO Express secrets.** Same page, but the **Secrets** tab this time (**not** Variables: variables print in
public logs, secrets never do). Click **New repository secret** twice:

| Name | Value |
| --- | --- |
| `ISO_NE_USERNAME` | your ISO Express username (Step 0d) |
| `ISO_NE_PASSWORD` | your ISO Express password |

Type the names exactly: all capitals, with underscores. **When you're done:** the Variables tab lists 6 names, and the
Secrets tab lists 2.

# Step 9 · Deploy and watch your bucket fill (15 minutes)

### 9a. Run the four deploy workflows
In your fork, open the **Actions** tab. In the left list, click each workflow, then **Run workflow** (branch: `main`) →
**Run workflow**:
1. **Deploy scraper-weather-obs**
2. **Deploy scraper-weather-forecast**
3. **Deploy scraper-isone**
4. **Deploy forecast-24h**

Each takes about 3 to 5 minutes. **Four green checks** = everything is deployed and scheduled.

On a brand-new project the four can bump into each other the first time (errors like `409 Could not create bucket
gcf-v2-uploads...` or `There were concurrent policy changes`). The workflows retry by themselves; if one still ends with a
red X, open it and click **Re-run failed jobs**. It will go green.

### 9b. Read the smoke tests
Each workflow's last step, **Smoke invoke (authenticated)**, runs the function once and prints what it did. Open each run:

| Workflow | Its last step should say |
| --- | --- |
| Deploy scraper-weather-obs | `"status": "ok", "hours_this_run": 6, ...` (6 or 7) |
| Deploy scraper-weather-forecast | `"status": "ok", "rows_this_run": 25, ...` |
| Deploy scraper-isone | `"status": "ok", "metered_hours_seen": ..., "five_min_complete_hours": ...` |
| Deploy forecast-24h | `"status": "no_model"`: **that's correct** until you upload a model in Lab 5 |

`scraper-isone` says `"skipped"`? Your secrets are missing or misspelled (Step 8c). Says `401`? The username or password is
wrong. Fix the secret, then re-run **Deploy scraper-isone**.

### 9c. Confirm in Cloud Shell

In [ ]:
gcloud functions list --regions="${REGION}" --format="table(name,state)"
gcloud scheduler jobs list --location="${REGION}" --format="table(name.basename(),schedule,state)"

**What you should see:** four functions `ACTIVE`, and four jobs `ENABLED`:
```
NAME                       STATE
forecast-24h               ACTIVE
scraper-isone              ACTIVE
scraper-weather-forecast   ACTIVE
scraper-weather-obs        ACTIVE

NAME                             SCHEDULE      STATE
forecast-24h-hourly              20 * * * *    ENABLED
scraper-isone-hourly             15 * * * *    ENABLED
scraper-weather-forecast-hourly  5 * * * *     ENABLED
scraper-weather-obs-hourly       5 * * * *     ENABLED
```

### 9d. Look at your data
In the Cloud Console: **Cloud Storage → Buckets →** your bucket → **`energy/datasets/`**. (The smoke tests already wrote the
first rows; every hour adds more.)

| File | What's in it |
| --- | --- |
| `bdl_weather_obs_master.csv` | one row per hour of Bradley weather |
| `bdl_weather_forecast_master.csv` | every forecast snapshot: 25 rows per hour (this hour + the next 24) |
| `ct_load_hourly_master.csv` | official metered Connecticut load (arrives 2 to 3 days late, so it starts a few days back) |
| `ct_load_5min_hourly_master.csv` | live Connecticut load, averaged per hour (about an hour behind) |
| `isone_ct_forecast_master.csv` | ISO New England's own Connecticut forecast, every version it publishes |

Click any file, then **Download**, to open it in Excel. The untouched downloads are kept in `energy/raw/`.

**You're done.** Your pipeline now collects weather and electricity data every hour on its own. Lab 5 trains a model and
uploads it; from then on `forecast-24h` writes your forecast for the next 24 hours to `energy/forecasts/latest.csv` every hour.

**Account safety:** GitHub only *deploys* your code. The schedule lives on Google Cloud, and nothing in this setup commits to
your GitHub repo on a timer. Keep it that way: don't add workflows that push to your repo on a schedule.

# Troubleshooting
Every one of these came up while testing this guide.

| What you see | What it means | Fix |
| --- | --- | --- |
| `number: )` is blank in Step 1 | Wrong project ID, or Cloud Shell not authorized | Check the ID (project picker → ID column), click **Authorize**, paste Step 1 again |
| `command not found` or empty values in Steps 2-8 | Cloud Shell restarted and wiped your variables | Paste the Step 1 cell again, then the step you were on |
| The Actions tab shows *"Workflows aren't being run on this forked repository"* | Forks start with Actions off | Click **"I understand my workflows, go ahead and enable them"** |
| A deploy fails with `409 Could not create bucket gcf-v2-uploads...` | Two deploys started at the same moment on a new project | It retries by itself; otherwise **Re-run failed jobs** |
| A deploy fails with `There were concurrent policy changes` | Same thing: two deploys edited permissions at once | It retries by itself; otherwise **Re-run failed jobs** |
| Deploy fails at **Auth (WIF)** with `Permission 'iam.serviceAccounts.getAccessToken' denied` | `GITHUB_REPO` in Step 1 doesn't match your fork exactly, or the workflow ran from a branch that isn't `main` | Paste Step 1 with the exact repo name, then Step 4 again; run the workflow on `main` |
| Deploy fails with *"RUNTIME_SA is not set"* or a blank project | A missing or misspelled GitHub **variable** | Compare the Variables tab with the Step 8a printout |
| `scraper-isone` says `"skipped"` | No ISO secrets yet, or they're under Variables instead of Secrets | Step 8c, then re-run **Deploy scraper-isone** |
| `scraper-isone` says `401` | Wrong ISO Express username or password | Fix the secret (copy it exactly as you log in to iso-ne.com), re-run |
| `forecast-24h` says `"no_model"` | Normal until Lab 5 | Nothing to fix |

# How it fits together
```
GitHub Actions (your fork, main branch)
   │  logs in with Workload Identity (no keys)
   ▼
cf-deployer ── deploys ──▶ 4 Cloud Functions (run as cf-runtime) ──▶ gs://YOUR-BUCKET/energy/
                                   ▲
                Cloud Scheduler ───┘  every hour (:05, :05, :15, :20 Eastern)
```
**Dr. Dave Wanik and Rohit Akole** · Dept. of Operations and Information Management · University of Connecticut ·
Fall 2026 edition, tested end to end on a brand-new project on 10/20/2026.